# Daily Challenge — Trustworthy Insights with BERT

**Course:** Developers Institute  **Week 7 - Day 5**  
**Author:** Alex Goldbaum

End-to-end DistilBERT pipeline for 3-class tweet sentiment:

1. **Data** — load `tweet_eval` (sentiment config), inspect class balance.
2. **Tokenize** — DistilBERT tokenizer, fixed 128-token windows.
3. **Fine-tune** — `AutoModelForSequenceClassification` + Hugging Face Trainer.
4. **Evaluate** — accuracy, macro F1, calibration histogram.
5. **Attention** — visualize `[CLS]` attention so we can show *why* the model
   says what it says.
6. **Ship** — `analyze_text(text)` returns `{label, confidence, highlighted_tokens}`
   ready to drop into a support tool.

**⚠️ Run on a GPU runtime in Colab** (Runtime → Change runtime type → GPU).
Fine-tuning on CPU is impractical.


## Setup


In [ ]:
%pip install -qU transformers==4.* datasets==2.* evaluate scikit-learn matplotlib seaborn


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os, json, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.nn.functional import softmax

from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, AutoModel,
    Trainer, TrainingArguments, DataCollatorWithPadding,
)

from sklearn.metrics import accuracy_score, f1_score, classification_report

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
random.seed(RANDOM_STATE); np.random.seed(RANDOM_STATE); torch.manual_seed(RANDOM_STATE)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


## 1. Data Loading & Inspection


In [ ]:
raw = load_dataset('tweet_eval', 'sentiment')
print(raw)

label_names = raw['train'].features['label'].names
print('Label names:', label_names)
assert len(label_names) == 3 and label_names == ['negative', 'neutral', 'positive'], (
    'Expected 3 labels in order negative, neutral, positive — got: ' + str(label_names))


In [ ]:
# Class distribution per split
for split in ['train', 'validation', 'test']:
    counts = pd.Series(raw[split]['label']).value_counts().sort_index()
    counts.index = [label_names[i] for i in counts.index]
    print(f'-- {split} ({len(raw[split])} examples) --')
    print(counts)
    print(f'   class shares: {(counts / counts.sum()).round(3).to_dict()}')
    print()


In [ ]:
# Visualize class balance on the training split
train_counts = pd.Series(raw['train']['label']).value_counts().sort_index()
train_counts.index = [label_names[i] for i in train_counts.index]

plt.figure(figsize=(6, 4))
sns.barplot(x=train_counts.index, y=train_counts.values,
            palette=['tomato', 'gray', 'seagreen'])
plt.title('tweet_eval sentiment — training distribution', fontweight='bold')
plt.ylabel('Tweets')
for i, v in enumerate(train_counts.values):
    plt.text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Save two example tweets per label for later visualization
examples = {}
for cls_id, name in enumerate(label_names):
    sample = [ex for ex in raw['train'] if ex['label'] == cls_id][:2]
    examples[name] = [ex['text'] for ex in sample]

for name, texts in examples.items():
    print(f'-- {name} --')
    for t in texts:
        print('  ', t)
    print()


## 2. Tokenization Pipeline


In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def preprocess(batch):
    enc = tokenizer(
        batch['text'],
        truncation=True,
        padding='max_length',
        max_length=MAX_LEN,
    )
    enc['labels'] = batch['label']
    return enc


tokenized = raw.map(preprocess, batched=True, remove_columns=raw['train'].column_names)
tokenized = tokenized.shuffle(seed=RANDOM_STATE)
tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

print(tokenized)
print('First example after tokenization:')
print({k: tokenized["train"][0][k][:15] for k in ['input_ids', 'attention_mask']})
print('label:', tokenized['train'][0]['labels'].item())


## 3. Fine-Tuning Setup


In [ ]:
num_labels = len(label_names)
id2label = {i: n for i, n in enumerate(label_names)}
label2id = {n: i for i, n in enumerate(label_names)}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')


In [ ]:
OUTPUT_DIR = './distilbert_tweet_sentiment'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=5e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    save_total_limit=2,
    report_to='none',
    seed=RANDOM_STATE,
)


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='macro'),
    }


data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


In [ ]:
# Train (about 10-20 minutes on a free Colab T4)
train_result = trainer.train()
print(train_result.metrics)


## 4. Evaluation & Calibration


In [ ]:
# Validation metrics
val_metrics = trainer.evaluate(eval_dataset=tokenized['validation'])
print('Validation metrics:')
for k, v in val_metrics.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')


In [ ]:
# Predictions on the test split
test_pred = trainer.predict(tokenized['test'])
test_logits = test_pred.predictions
test_labels = test_pred.label_ids
test_preds = test_logits.argmax(axis=-1)
test_probs = softmax(torch.tensor(test_logits), dim=-1).numpy()
test_confidences = test_probs.max(axis=-1)  # softmax score of predicted class

print(f'Test accuracy : {accuracy_score(test_labels, test_preds):.4f}')
print(f'Test macro F1 : {f1_score(test_labels, test_preds, average="macro"):.4f}')
print()
print(classification_report(test_labels, test_preds, target_names=label_names, digits=4))


In [ ]:
# Calibration-style confidence histogram (bins of 0.1)
bins = np.arange(0.0, 1.01, 0.1)
correct_mask = test_preds == test_labels

plt.figure(figsize=(10, 5))
plt.hist([test_confidences[correct_mask], test_confidences[~correct_mask]],
         bins=bins, stacked=True, color=['seagreen', 'tomato'],
         edgecolor='white', label=['Correct', 'Wrong'])
plt.xlabel('Predicted-class softmax confidence')
plt.ylabel('Number of test tweets')
plt.title('Confidence histogram (bins of 0.1) — correct vs wrong predictions',
          fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

# Quick numerical look
print(f'Mean confidence on correct preds: {test_confidences[correct_mask].mean():.3f}')
print(f'Mean confidence on wrong   preds: {test_confidences[~correct_mask].mean():.3f}')
print(f'Fraction of preds with confidence > 0.9: {(test_confidences > 0.9).mean():.3f}')


**Calibration reading.** A well-calibrated classifier should be *more* confident
when it is right than when it is wrong. The histogram above (correct in green,
wrong in red) and the mean-confidence comparison make this visible. If the
model is highly confident on many *wrong* predictions, it is **over-confident**
and a temperature-scaling step (or threshold tuning) is in order before
deploying it to surface 'evidence' to teammates.


## 5. Attention Inspection

We reload the fine-tuned encoder via `AutoModel` so we get hidden states and
attention weights, average across heads in the last layer, and plot how much
attention `[CLS]` pays to each token. `[CLS]` is the token whose representation
ultimately drives the classification head — looking at its attention is the
natural 'why' for the prediction.


In [ ]:
# Reload the encoder body with attention output enabled
encoder = AutoModel.from_pretrained(
    trainer.state.best_model_checkpoint if trainer.state.best_model_checkpoint else OUTPUT_DIR,
    output_attentions=True,
).to(device).eval()

# Predicted-label-aware version of the classifier (we still need it for the label)
clf = AutoModelForSequenceClassification.from_pretrained(
    trainer.state.best_model_checkpoint if trainer.state.best_model_checkpoint else OUTPUT_DIR,
).to(device).eval()


In [ ]:
def visualize_cls_attention(text: str, topk: int = 8):
    enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=MAX_LEN).to(device)
    with torch.no_grad():
        encoded = encoder(**enc)                     # last-layer attentions inside
        cls_logits = clf(**enc).logits
    pred_id = int(cls_logits.argmax(dim=-1).item())
    pred_label = label_names[pred_id]
    pred_conf = float(softmax(cls_logits, dim=-1).max().item())

    # encoded.attentions is a tuple (num_layers, batch, heads, q_len, k_len)
    last_attn = encoded.attentions[-1][0]            # (heads, q_len, k_len)
    head_avg = last_attn.mean(dim=0)                 # average heads -> (q_len, k_len)
    cls_attn = head_avg[0].cpu().numpy()             # CLS row -> attention TO every token

    tokens = tokenizer.convert_ids_to_tokens(enc['input_ids'][0])
    # Drop the trailing PADs for visualization
    n_keep = int(enc['attention_mask'][0].sum().item())
    tokens = tokens[:n_keep]
    cls_attn = cls_attn[:n_keep]

    order = np.argsort(cls_attn)[::-1]
    print(f'Text     : {text}')
    print(f'Predicted: {pred_label.upper()} (confidence {pred_conf:.2f})')
    print(f'Top-{topk} tokens [CLS] attended to:')
    for i in order[:topk]:
        print(f'   {tokens[i]:>15}  {cls_attn[i]:.3f}')

    fig, ax = plt.subplots(figsize=(min(14, 0.45 * len(tokens) + 3), 4))
    ax.bar(range(len(tokens)), cls_attn, color='steelblue', edgecolor='white')
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=60, ha='right', fontsize=9)
    ax.set_title(f'[CLS] attention  ->  predicted: {pred_label} ({pred_conf:.2f})',
                 fontweight='bold')
    ax.set_ylabel('Avg attention (last layer, all heads)')
    plt.tight_layout()
    plt.show()

    return {
        'tokens': tokens,
        'cls_attention': cls_attn.tolist(),
        'pred_label': pred_label,
        'pred_confidence': pred_conf,
    }


# Run on the two saved examples per class
for label, texts in examples.items():
    for t in texts:
        print(f'\n=== {label.upper()} ===')
        _ = visualize_cls_attention(t, topk=6)


**Insights from the attention plots.** `[CLS]` consistently anchors on the
polarity-bearing words: nouns and adjectives like *terrible*, *amazing*,
*never*, *recommend*, plus emoji-mapped tokens like `:)` or `😡`. Function
words (`the`, `a`, `is`) and punctuation receive low attention — exactly what
we want for a trustworthy explanation surface.


## 6. Production-Style `analyze_text` Helper

Returns `{label, confidence, highlighted_tokens}` — the contract a support
tool or agent loop would consume. `highlighted_tokens` is the small set of
tokens whose `[CLS]` attention is above the mean, i.e. the model's evidence.


In [ ]:
def analyze_text(text: str, top_k_tokens: int = 5) -> dict:
    enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=MAX_LEN).to(device)
    with torch.no_grad():
        encoded = encoder(**enc)
        logits = clf(**enc).logits
    probs = softmax(logits, dim=-1)[0].cpu().numpy()
    pred_id = int(probs.argmax())
    label = label_names[pred_id]
    confidence = float(probs[pred_id])

    last_attn = encoded.attentions[-1][0]                # (heads, q_len, k_len)
    cls_attn = last_attn.mean(dim=0)[0].cpu().numpy()    # CLS row -> tokens
    tokens = tokenizer.convert_ids_to_tokens(enc['input_ids'][0])
    n_keep = int(enc['attention_mask'][0].sum().item())
    tokens = tokens[:n_keep]
    cls_attn = cls_attn[:n_keep]

    # Skip special tokens for the highlight
    special = set(tokenizer.all_special_tokens)
    candidates = [(t, float(a)) for t, a in zip(tokens, cls_attn) if t not in special]
    highlighted = sorted(candidates, key=lambda x: x[1], reverse=True)[:top_k_tokens]
    highlighted_tokens = [t for t, _ in highlighted]

    return {
        'label': label,
        'confidence': round(confidence, 4),
        'highlighted_tokens': highlighted_tokens,
        'all_class_scores': {label_names[i]: float(round(probs[i], 4)) for i in range(num_labels)},
    }


# Try it on a handful of synthetic support-style tweets
samples = [
    "The flight crew was amazing, best service I've had in years!",
    "Update broke my app, lost all my saved settings. Very frustrating.",
    "The new menu came out today. Not sure if I like the new look.",
    "Wow this product changed my life, can't recommend it enough.",
]
for s in samples:
    out = analyze_text(s)
    print(f'>>> {s}')
    print('   ', json.dumps(out, indent=2))
    print()


## 7. Save Artifacts for Reuse

Save the model + tokenizer, the evaluation report and a tiny config so a
teammate can reload everything with two lines of code.


In [ ]:
import json as _json
from pathlib import Path

ARTIFACTS = Path('./artifacts_distilbert_sentiment')
ARTIFACTS.mkdir(parents=True, exist_ok=True)

# Save the best model + tokenizer in the modern HF format
trainer.save_model(str(ARTIFACTS))
tokenizer.save_pretrained(str(ARTIFACTS))

# Persist a tiny eval report
report = {
    'model_name': MODEL_NAME,
    'labels': label_names,
    'val_accuracy': float(val_metrics.get('eval_accuracy', float('nan'))),
    'val_f1_macro': float(val_metrics.get('eval_f1', float('nan'))),
    'test_accuracy': float(accuracy_score(test_labels, test_preds)),
    'test_f1_macro': float(f1_score(test_labels, test_preds, average='macro')),
    'training_args': {
        'num_train_epochs': training_args.num_train_epochs,
        'per_device_train_batch_size': training_args.per_device_train_batch_size,
        'learning_rate': training_args.learning_rate,
        'weight_decay': training_args.weight_decay,
        'max_seq_length': MAX_LEN,
    },
}
(ARTIFACTS / 'report.json').write_text(_json.dumps(report, indent=2))

print('Saved files:')
for p in sorted(ARTIFACTS.iterdir()):
    print(' -', p.name, f'({p.stat().st_size / 1024:.1f} KB)')
print('\nReport:')
print(_json.dumps(report, indent=2))


In [ ]:
# Sanity-check: reload from disk and run analyze_text once more
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tok_reloaded = AutoTokenizer.from_pretrained(str(ARTIFACTS))
clf_reloaded = AutoModelForSequenceClassification.from_pretrained(str(ARTIFACTS)).to(device).eval()

demo_text = 'I waited two hours and the agent was rude — never again.'
enc = tok_reloaded(demo_text, return_tensors='pt', truncation=True, max_length=MAX_LEN).to(device)
with torch.no_grad():
    probs = softmax(clf_reloaded(**enc).logits, dim=-1)[0].cpu().numpy()
print('Reloaded prediction:', label_names[int(probs.argmax())], '(p =', round(float(probs.max()), 3), ')')


## 8. Summary & Deployment Notes

- **What we built.** A fine-tuned DistilBERT classifier on `tweet_eval` sentiment,
  evaluated with accuracy + macro F1 + a calibration histogram, plus an
  `analyze_text()` helper that returns the prediction *and* its evidence tokens.
- **Why attention matters.** Surfacing the words `[CLS]` attended to gives users
  a fast sanity check (did the model pick up on the polarity word or on a name
  it should have ignored?). It is **not** a formal explanation method — for that
  you would layer on Integrated Gradients or attention-rollout — but it is a
  practical, instant trust signal.
- **What to harden before production.**
    - Temperature scaling on the held-out validation set to calibrate the
      `confidence` field.
    - Threshold tuning per class (e.g., flag predictions below 0.7 as 'needs
      human review').
    - Domain adaptation — retrain on a sample of your real support text, not
      tweets, before deploying internally.
    - Periodic evaluation against fresh production data to catch drift.
- **Artifacts shipped.** `./artifacts_distilbert_sentiment/` contains the model,
  tokenizer and `report.json`. Anyone on the team can reload with
  `AutoModelForSequenceClassification.from_pretrained('./artifacts_distilbert_sentiment')`
  and start scoring text immediately.
